In [1]:
# Prophet Model for Kelani River Water Level Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from prophet import Prophet
from prophet.plot import plot_plotly, plot_components_plotly
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set(rc={'figure.figsize':(15, 8)})

def fit_prophet_model(train, test, yearly=True, weekly=True, daily=False, changepoint_prior_scale=0.05):
    """
    Fit Prophet model and make forecasts
    """
    # Convert to Prophet format
    train_prophet = pd.DataFrame({
        'ds': train.index,
        'y': train.values
    })
    
    # Create and fit model
    model = Prophet(
        seasonality_mode='multiplicative',
        yearly_seasonality=yearly,
        weekly_seasonality=weekly,
        daily_seasonality=daily,
        changepoint_prior_scale=changepoint_prior_scale
    )
    
    # Add additional seasonalities if needed
    # model.add_seasonality(name='monthly', period=30.5, fourier_order=5)
    
    model.fit(train_prophet)
    
    # Create future dataframe for prediction period
    future = model.make_future_dataframe(periods=len(test), freq='D')
    
    # Generate forecast
    forecast = model.predict(future)
    
    # Extract relevant part for test period
    test_forecast = forecast.iloc[-len(test):].copy()
    
    # Create series for easier comparison
    forecast_series = pd.Series(test_forecast['yhat'].values, index=test.index)
    
    # Visualize the forecast
    plt.figure(figsize=(15, 8))
    plt.plot(train[-30:], label='Training Data (last 30 points)')
    plt.plot(test, label='Actual')
    plt.plot(forecast_series, label='Forecast (yhat)')
    
    # Add uncertainty intervals
    plt.fill_between(
        test.index, 
        test_forecast['yhat_lower'].values, 
        test_forecast['yhat_upper'].values, 
        color='grey', 
        alpha=0.2, 
        label='95% Confidence Interval'
    )
    
    plt.title('Prophet Forecast vs Actual')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    # Plot Prophet components
    fig = model.plot_components(forecast)
    plt.tight_layout()
    plt.show()
    
    # Make sure both test and forecast don't have NaN values
    valid_idx = ~(np.isnan(test) | np.isnan(forecast_series))
    
    if sum(valid_idx) < 2:
        print("WARNING: Not enough valid data points for evaluation.")
        return model, forecast, forecast_series, {'rmse': np.nan, 'mae': np.nan, 'r2': np.nan}
    
    # Filter out NaN values
    test_clean = test[valid_idx]
    forecast_clean = forecast_series[valid_idx]
    
    # Calculate metrics
    try:
        rmse = np.sqrt(mean_squared_error(test_clean, forecast_clean))
        mae = mean_absolute_error(test_clean, forecast_clean)
        r2 = r2_score(test_clean, forecast_clean)
        
        print(f'RMSE: {rmse:.4f}')
        print(f'MAE: {mae:.4f}')
        print(f'R²: {r2:.4f}')
        
        return model, forecast, forecast_series, {'rmse': rmse, 'mae': mae, 'r2': r2}
    except Exception as e:
        print(f"Error calculating metrics: {str(e)}")
        return model, forecast, forecast_series, {'rmse': np.nan, 'mae': np.nan, 'r2': np.nan}

def find_best_prophet_model(train, test):
    """
    Try different Prophet configurations and return the best one
    """
    prophet_configs = [
        {"name": "Default", "changepoint_prior_scale": 0.05, "yearly": True, "weekly": True, "daily": False},
        {"name": "High Flexibility", "changepoint_prior_scale": 0.5, "yearly": True, "weekly": True, "daily": False},
        {"name": "Low Flexibility", "changepoint_prior_scale": 0.01, "yearly": True, "weekly": True, "daily": False},
        {"name": "Weekly Only", "changepoint_prior_scale": 0.05, "yearly": False, "weekly": True, "daily": False},
        {"name": "Yearly Only", "changepoint_prior_scale": 0.05, "yearly": True, "weekly": False, "daily": False}
    ]
    
    best_model = None
    best_forecast = None
    best_forecast_series = None
    best_metrics = {"rmse": float('inf')}
    best_config = None
    
    for config in prophet_configs:
        print(f"\nTrying {config['name']}...")
        try:
            model, forecast, forecast_series, metrics = fit_prophet_model(
                train, 
                test, 
                yearly=config["yearly"],
                weekly=config["weekly"],
                daily=config["daily"],
                changepoint_prior_scale=config["changepoint_prior_scale"]
            )
            
            print(f"{config['name']} - RMSE: {metrics['rmse']:.4f}, MAE: {metrics['mae']:.4f}, R²: {metrics['r2']:.4f}")
            
            # Keep track of best model
            if not np.isnan(metrics['rmse']) and metrics['rmse'] < best_metrics['rmse']:
                best_model = model
                best_forecast = forecast
                best_forecast_series = forecast_series
                best_metrics = metrics
                best_config = config
        except Exception as e:
            print(f"Error fitting {config['name']}: {str(e)}")
    
    if best_model is not None:
        print(f"\nBest Prophet Model: {best_config['name']}")
        print(f"Parameters: {best_config}")
        print(f"RMSE: {best_metrics['rmse']:.4f}")
        print(f"MAE: {best_metrics['mae']:.4f}")
        print(f"R²: {best_metrics['r2']:.4f}")
    
    return best_config['name'], best_model, best_forecast, best_forecast_series, best_metrics

def analyze_water_level_station_prophet(file_path, station_name):
    """
    Process, analyze and build Prophet model for water level station data
    """
    print(f"\n{'='*80}")
    print(f"Analyzing station: {station_name}")
    print(f"{'='*80}")
    
    # Read data
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"Error reading file {file_path}: {str(e)}")
        return None
    
    # Check if data exists
    if df.empty:
        print(f"No data found for {station_name}")
        return None
    
    # Parse dates and set as index
    try:
        df.columns = ['datetime', 'unit', 'water_level', 'label']
        df['datetime'] = pd.to_datetime(df['datetime'])
        df.set_index('datetime', inplace=True)
    except Exception as e:
        print(f"Error parsing dates: {str(e)}")
        print("Columns in file:", df.columns)
        return None
    
    # Display basic info
    print("\nData Overview:")
    print(f"Time Range: {df.index.min()} to {df.index.max()}")
    print(f"Number of records: {len(df)}")
    print(f"Number of missing values: {df['water_level'].isna().sum()}")
    
    # Handle missing values if any
    if df['water_level'].isna().any():
        df['water_level'] = df['water_level'].interpolate(method='time')
    
    # Plot water level data
    plt.figure(figsize=(15, 6))
    plt.plot(df['water_level'])
    plt.title(f'Water Level - {station_name}')
    plt.xlabel('Date')
    plt.ylabel('Water Level (m)')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    # Resample to daily frequency (using mean) to reduce noise
    daily_data = df['water_level'].resample('D').mean()
    
    # Handle any remaining NaN values in daily data
    if daily_data.isna().any():
        print(f"Filling {daily_data.isna().sum()} NaN values in daily data")
        daily_data = daily_data.interpolate(method='time')
    
    # Split data into train and test sets
    train_size = int(len(daily_data) * 0.8)
    train, test = daily_data[:train_size], daily_data[train_size:]
    
    print(f"\nTraining set size: {len(train)}")
    print(f"Test set size: {len(test)}")
    
    # Find best Prophet model
    print("\nFinding best Prophet model...")
    model_name, model, forecast, forecast_series, metrics = find_best_prophet_model(train, test)
    
    return {
        'station_name': station_name,
        'model_name': model_name,
        'model': model,
        'forecast': forecast_series,
        'metrics': metrics
    }

# Main execution
def main():
    base_dir = r"c:\Users\RushenSamodya\Documents\GitHub\Final_Year_Project\labeled_data"
    
    # Path to Kelani River water level data
    kelani_dir = os.path.join(base_dir, "water_level_data", "kelani_ganga")
    
    # Check if directory exists
    if not os.path.exists(kelani_dir):
        print(f"Directory does not exist: {kelani_dir}")
        return
    
    # Get all stations for Kelani River
    stations = [f for f in os.listdir(kelani_dir) if f.endswith('.csv')]
    
    if not stations:
        print(f"No station data found in {kelani_dir}")
        return
    
    print(f"Found {len(stations)} stations for Kelani River")
    
    # Results container
    results = []
    
    # Process each station
    for station_file in stations:
        station_path = os.path.join(kelani_dir, station_file)
        station_name = station_file.replace('.csv', '')
        
        # Analyze station data
        result = analyze_water_level_station_prophet(station_path, station_name)
        
        if result:
            results.append(result)
    
    # Compare results across stations
    if results:
        print("\n\nComparison of model performance across stations:")
        comparison = pd.DataFrame([
            {
                'Station': r['station_name'],
                'Best Prophet Model': r['model_name'],
                'RMSE': r['metrics']['rmse'],
                'MAE': r['metrics']['mae'],
                'R²': r['metrics']['r2']
            }
            for r in results
        ])
        
        print(comparison)
        
        # Save results to CSV
        comparison.to_csv('prophet_model_results.csv', index=False)
        print("Results saved to prophet_model_results.csv")

# Execute main function
if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'prophet'